In [1]:
##############
## INITIATE ##
##############

## Imports
import os
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D
import pandas as pd

## Figure parameters
plt.rcParams['figure.dpi'] = 300
figsize = (14,5)
custom_params = {'axes.spines.right': False, 
                 'axes.spines.top': False, 
                 'axes.facecolor':'whitesmoke'
                }
sns.set_theme(style="ticks", rc=custom_params)

#
###

In [2]:
###############
## FUNCTIONS ##
###############

def normalise(x):
    """
    """
    return (x - x.min())/(x.max() - x.min())

def unique_preserve_order(seq):
    """
    """
    seen = set()
    unique_seq = []
    for item in seq:
        if item not in seen:
            seen.add(item)
            unique_seq.append(item)
    return unique_seq

def bin_y_along_x(arr_x, x_index, arr_y, y_index):
    """
    """
    x_unique = np.unique(arr_x[:,x_index])
    y_binned = []
    for x_unique_ in x_unique:
        s = np.argwhere(arr_x[:,x_index] == x_unique_)
        y_binned += [np.mean(arr_y[s,y_index])]
    y_binned = np.array(y_binned)
    return x_unique, y_binned  

def collapse_columns_2Darray_to_string(arr, sep=""):
    """
    """
    arr_str = []
    for arr_ in arr:
        arr_ = [arr__ for arr__ in arr_.astype(str)]
        arr_ = sep.join(arr_)
        arr_str += [arr_]
    return np.array(arr_str)

def load_data(pwd, fold_number):
    """
    Loads hyperparameter data for a specific fold from the given directory.

    Parameters:
    pwd (str): The path to the directory containing the data.
    fold_number (int): The fold number to load data for.

    Returns:
    tuple: Arrays of hyperparameters, train AUCs, validation AUCs, and test AUCs.
    """
    pti = os.path.join(pwd, f"hypertrain_fold_{fold_number}/")
    ht = pickle.load(open(os.path.join(pti, "obj_hypertrain_list.pkl"), "rb"))
    
    hp, train, val, test = [], [], [], []
    for ht_ in ht:
        hp.append(ht_[:5])
        train.append(ht_[5])
        val.append(ht_[6])
        test.append(ht_[7])
        
    return np.array(hp), np.array(train), np.array(val), np.array(test)

def load_fold_data(pwd, folds):
    """
    """
    hp, train, val, test = [], [], [], []
    for fold in folds:
        fold_hp, fold_train, fold_val, fold_test = load_data(pwd, fold)
        hp.append(fold_hp)
        train.append(fold_train)
        val.append(fold_val)
        test.append(fold_test)
    return np.concatenate(hp), np.concatenate(train), np.concatenate(val), np.concatenate(test)

def compute_unique_hyperparameter_sets(hp_data, column_indices, separator="_"):
    """Compute unique combinations of hyperparameters and their indices."""
    hp_keys = collapse_columns_2Darray_to_string(hp_data[:, column_indices], sep=separator)
    hp_keys_unique = np.unique(hp_keys)
    hp_set_index = np.arange(len(hp_keys_unique)) + 1
    return hp_keys, hp_keys_unique, hp_set_index

def assemble_data_by_hyperparameter_sets(hp_keys, hp_keys_unique, target_array):
    """Assemble validation AUCs grouped by unique hyperparameter sets."""
    x_data = []
    y_data = []
    for hp_keys_unique_ in hp_keys_unique:
        s = np.argwhere(hp_keys == hp_keys_unique_)
        x_data.append(hp_keys_unique_)
        y_data.append(target_array.mean(1)[s].flatten())
    return x_data, y_data

def remove_nan_from_data(y_data):
    """Remove NaN values from the AUC data."""
    return [arr[~np.isnan(arr)] for arr in y_data]

def plot_boxplot(y_data):
    """Plot a boxplot of the validation AUCs for each hyperparameter set."""
    plt.boxplot(y_data)
    plt.xlabel("Hyperparameter Set")
    plt.ylabel("Validation AUC")
    plt.title("Comparison of validation AUC of hyperparameter sets")
    plt.tight_layout()
    plt.show()
    plt.close()

def compute_and_plot_mean_auc(y_data, hp_set_index, plot=False):
    """Compute and plot the mean AUC for each hyperparameter set."""
    auc_list = [np.mean(y_) for y_ in y_data]
    if plot == True:
        plt.scatter(hp_set_index, auc_list)
        plt.plot([0, len(hp_set_index)], [0, 0], linestyle="--", color="black")
        for i in range(len(auc_list)):
            plt.plot([hp_set_index[i], hp_set_index[i]], [0, auc_list[i]])
    
        plt.xlim(1-0.5, len(hp_set_index)+0.5)
        plt.xlabel("Hyperparameter Set")
        plt.ylabel("Mean Validation AUC")
        plt.title("Comparison of validation AUC of hyperparameter sets")
        plt.tight_layout()
        plt.show()
        plt.close()
    return auc_list

def compute_and_plot_t_statistic(y_data, hp_set_index, plot=False):
    """Compute and plot the t-statistic for each hyperparameter set."""
    mean_y_data = np.mean(np.concatenate(y_data))
    std_y_data = np.std(np.concatenate(y_data))
    t_list = [(np.mean(y_) - mean_y_data) / std_y_data for y_ in y_data]

    if plot == True:
        plt.scatter(hp_set_index, t_list)
        plt.plot([0, len(hp_set_index)], [0, 0], linestyle="--", color="black")
        for i in range(len(t_list)):
            plt.plot([hp_set_index[i], hp_set_index[i]], [0, t_list[i]])
    
        plt.xlim(1-0.5, len(hp_set_index)+0.5)
        plt.xlabel("Hyperparameter Set")
        plt.ylabel("t-statistic of Validation AUC")
        plt.title("Comparison of validation t-statistics of hyperparameter sets")
        plt.tight_layout()
        plt.show()
        plt.close()
        
    return t_list

def identify_best_hyperparameter_set(t_list, hp_keys_unique):
    """Identify the best hyperparameter set based on the t-statistic."""
    hp_star_index = np.argmax(t_list)
    hp_star = hp_keys_unique[hp_star_index]
    return hp_star_index + 1, hp_star

def load_and_process_folds(pwd, folds, column_indices, plot=False):
    """Load data from multiple folds and process hyperparameter sets and AUCs."""
    hp, train, val, test = load_fold_data(pwd, folds)

    # Compute unique hyperparameter sets
    hp_keys, hp_keys_unique, hp_set_index = compute_unique_hyperparameter_sets(hp, column_indices)

    # Assemble data by hyperparameter sets
    x_data, y_data = assemble_data_by_hyperparameter_sets(hp_keys, hp_keys_unique, val)

    # Remove NaNs from data
    y_data = remove_nan_from_data(y_data)

    # Plot boxplot
    if plot == True:
        plot_boxplot(y_data)

    # Compute and plot mean AUC
    auc_list = compute_and_plot_mean_auc(y_data, hp_set_index)

    # Compute and plot t-statistic
    t_list = compute_and_plot_t_statistic(y_data, hp_set_index)

    # Identify and print the best hyperparameter set
    hp_star_index, hp_star = identify_best_hyperparameter_set(t_list, hp_keys_unique)
    print(f"The best hyperparameter set is: {hp_star_index}")
    print(f"HP star is: {hp_star}")

    # Save the best hyperparameter set
    return hp_star

def load_and_process_folds_separately(pwd, folds, column_indices, plot=False):
    """Load and process each fold separately, generating plots and identifying the best hyperparameter set for each."""
    hp_star_all = []
    hp_all_folds = []
    train_all_folds = []
    val_all_folds = []
    test_all_folds = []

    for fold in folds:
        print(f"Processing fold {fold}...")

        # Load data for the specific fold
        hp, train, val, test = load_data(pwd, fold)

        # Store the loaded data
        hp_all_folds.append(hp)
        train_all_folds.append(train)
        val_all_folds.append(val)
        test_all_folds.append(test)

        # Compute unique hyperparameter sets
        hp_keys, hp_keys_unique, hp_set_index = compute_unique_hyperparameter_sets(hp, column_indices)

        # Assemble data by hyperparameter sets
        x_data, y_data = assemble_data_by_hyperparameter_sets(hp_keys, hp_keys_unique, val)

        # Remove NaNs from data
        y_data = remove_nan_from_data(y_data)

        # Plot boxplot
        if plot == True:
            plot_boxplot(y_data)

        # Compute and plot mean AUC
        auc_list = compute_and_plot_mean_auc(y_data, hp_set_index)

        # Compute and plot t-statistic
        t_list = compute_and_plot_t_statistic(y_data, hp_set_index)

        # Identify and print the best hyperparameter set
        hp_star_index, hp_star = identify_best_hyperparameter_set(t_list, hp_keys_unique)
        print(f"Best hyperparameter set for fold {fold} is: {hp_star_index}")
        print(f"HP star is: {hp_star}")

        # Save the best hyperparameter set for this fold
        hp_star_all.append(hp_star)

    return hp_star_all, hp_all_folds, train_all_folds, val_all_folds, test_all_folds

def plot_auc_violin(all_folds_data, labels, title):
    """Plot a violin plot comparing AUC data across different folds for each label."""
    plt.figure(figsize=(12, 6))
    fontsize = 12

    colors = ['grey', 'salmon', 'skyblue', 'limegreen', 'orange']  # Example colors for each fold
    data_offsets = np.linspace(-0.2, 0.2, len(all_folds_data))  # Spread data points to avoid overlap

    for i, (fold_data, color, offset) in enumerate(zip(all_folds_data, colors, data_offsets)):
        x = [1 + offset + j for j in range(len(fold_data))]
        parts = plt.violinplot(fold_data, positions=x, showmeans=False, showmedians=False, showextrema=False)
        for pc in parts['bodies']:
            pc.set_facecolor(color)
            pc.set_alpha(0.5)
        q05, quartile1, medians, quartile3, q95 = np.percentile(fold_data, [5, 25, 50, 75, 95], axis=1)
        inds = np.arange(1, len(medians) + 1) + offset
        means = np.mean(fold_data, axis=1)
        plt.scatter(inds, means, marker='o', color='red', s=30, zorder=3)
        plt.scatter(inds, medians, marker='o', color='white', s=30, zorder=3)
        plt.vlines(inds, quartile1, quartile3, color='k', linestyle='-', lw=5)
        plt.vlines(inds, q05, q95, color='k', linestyle='-', lw=1)

    # Add line at 0.5
    plt.axhline(y=0.5, linestyle='--', color='gray')

    # Legend
    custom_lines = [Line2D([0], [0], color=color, lw=2) for color in colors[:len(all_folds_data)]]
    plt.legend(custom_lines, [f'Fold {i+1}' for i in range(len(all_folds_data))], loc='lower right', frameon=False, fontsize=fontsize)

    # Graphical parameters
    sns.set(font_scale=1.2)
    plt.xticks(np.arange(1, len(labels) + 1), labels=labels, rotation='vertical', size=fontsize)
    plt.xlabel("")
    plt.ylabel("Accuracy of detection (AUROC)", size=fontsize)
    plt.ylim(0.25, 1)
    plt.xlim(0, len(labels) + 1)
    plt.title(title)
    plt.tight_layout()
    plt.show()
    plt.close()

def prepare_fold_data(hp_star, hp_keys, aucs):
    """Prepare the data for the best model across multiple folds."""
    indices = np.argwhere(hp_star == hp_keys).flatten()
    auc_star = aucs[indices]
    if len(auc_star.shape) == 1:
        auc_star = auc_star.reshape(-1, 1)
    return [auc_star[:, i] for i in range(auc_star.shape[1])]

def plot_auc_bars(all_folds_data, labels, title):
    """Plot bars comparing AUC data across different folds and labels."""
    plt.figure(figsize=(12, 6))
    fontsize = 12

    colors = ['grey', 'salmon', 'skyblue', 'limegreen', 'orange']  # Colors for each fold
    num_labels = len(set(labels))  # Calculate number of unique labels
    num_folds = len(all_folds_data[0])  # Number of folds (assumed to be the same for all labels)

    # x_positions for each label (centered across folds)
    x_positions = np.arange(1, num_labels + 1)

    for i, (fold_data, color) in enumerate(zip(all_folds_data, colors)):
        for j in range(num_folds):
            x = x_positions + (i - len(all_folds_data)/2) * 0.1  # Small offset to avoid overlap
            q05, quartile1, medians, quartile3, q95 = np.percentile(fold_data[j], [5, 25, 50, 75, 95])
            means = np.mean(fold_data[j])

            # Plot bars (vlines)
            plt.scatter(x, [means] * len(x), marker='o', color='red', s=30, zorder=3)
            plt.scatter(x, [medians] * len(x), marker='o', color='white', s=30, zorder=3)
            plt.vlines(x, [quartile1] * len(x), [quartile3] * len(x), color=color, linestyle='-', lw=5)
            plt.vlines(x, [q05] * len(x), [q95] * len(x), color=color, linestyle='-', lw=1)

    # Add line at 0.5
    plt.axhline(y=0.5, linestyle='--', color='gray')

    # Legend
    custom_lines = [Line2D([0], [0], color=color, lw=2) for color in colors[:len(all_folds_data)]]
    plt.legend(custom_lines, [f'Set {i+1}' for i in range(len(all_folds_data))], loc='lower right', frameon=False, fontsize=fontsize)

    # Graphical parameters
    sns.set(font_scale=1.2)
    plt.xticks(x_positions, labels[:num_labels], rotation='vertical', size=fontsize)
    plt.xlabel("")
    plt.ylabel("Accuracy of detection (AUROC)", size=fontsize)
    plt.ylim(0.25, 1)
    plt.xlim(0.5, num_labels + 0.5)
    plt.title(title)
    plt.tight_layout()
    plt.show()
    plt.close()

def collect_results_across_labels(master_folder, selected_folders, column_indices):
    """Collect AUC results and best hyperparameters across specific label folders in a predefined order.
    
    Args:
    - master_folder (str): The path to the master folder containing label folders.
    - selected_folders (list of str): The list of label folders in the order to process.
    - column_indices (list of int): The indices of the columns used for hyperparameter keys.
    
    Returns:
    - all_train_data (list): Aggregated train AUC data across all labels and folds.
    - all_val_data (list): Aggregated validation AUC data across all labels and folds.
    - all_test_data (list): Aggregated test AUC data across all labels and folds.
    - labels (list): The list of labels corresponding to the processed folders.
    - best_hyperparams (list of lists): A list where each element is a list of best hyperparameters for each fold, corresponding to the labels in the order provided.
    """
    all_train_data = []
    all_val_data = []
    all_test_data = []
    labels = []
    best_hyperparams = []

    # Iterate over each folder in the predefined order
    for label_folder in selected_folders:
        label_path = os.path.join(master_folder, label_folder)
        if os.path.isdir(label_path):  # Ensure it's a directory
            print(f"Processing label: {label_folder}")

            # Collect results for this label using the existing functions
            folds = [1, 2, 3, 4, 5]  # Assuming 5 folds
            hp_star_all_folds, hp_all_folds, train_all_folds, val_all_folds, test_all_folds = load_and_process_folds_separately(label_path, folds, column_indices)
            
            # Prepare data for all folds
            train_all_stars = []
            val_all_stars = []
            test_all_stars = []

            for i in range(len(hp_star_all_folds)):
                hp_star = hp_star_all_folds[i]
                hp_keys = collapse_columns_2Darray_to_string(hp_all_folds[i][:,2:5], sep="_")
                
                train_star = prepare_fold_data(hp_star, hp_keys, train_all_folds[i])
                val_star = prepare_fold_data(hp_star, hp_keys, val_all_folds[i])
                test_star = prepare_fold_data(hp_star, hp_keys, test_all_folds[i])
                
                train_all_stars.append(train_star)
                val_all_stars.append(val_star)
                test_all_stars.append(test_star)

            # Append the results for this label to the aggregate lists
            all_train_data.extend(train_all_stars)
            all_val_data.extend(val_all_stars)
            all_test_data.extend(test_all_stars)
            labels.extend([label_folder] * len(train_all_stars))
            
            # Store the best hyperparameters for this label
            best_hyperparams.append(hp_star_all_folds)

    return all_train_data, all_val_data, all_test_data, labels, best_hyperparams

def save_best_hyperparameters_to_csv(labels, best_hyperparams, output_file):
    """Saves the best hyperparameters for each label to a CSV file.
    
    Args:
    - labels (list of str): The list of labels.
    - best_hyperparams (list of lists): The best hyperparameters corresponding to each label.
    - output_file (str): The path to the output CSV file.
    """
    # Prepare the data for the DataFrame
    data = []
    for label, hyperparams in zip(labels, best_hyperparams):
        for fold_idx, hp in enumerate(hyperparams, start=1):
            data.append({"Label": label, "Fold": fold_idx, "Best Hyperparameters": hp})
    
    # Create a DataFrame
    df = pd.DataFrame(data)
    
    # Save to CSV
    df.to_csv(output_file, index=False)
    print(f"Best hyperparameters saved to {output_file}")
    
#
###

In [3]:
## Imports
import os
import pandas as pd
import pickle
import numpy as np
import time

##
import p_pawmil as pawmil

##
from p_pawmil.f_aggregator_pawmil import AttentionLastAggregator as AggregatorArchitecture
from p_pawmil.f_dataset_embeds import InstanceEmbeddingsDataset
from p_pawmil.f_dataset_embeds import GroupWeightedInstanceEmbeddingsDataset
from p_pawmil.f_train import train_model
from p_pawmil.f_train import train_model_weighted
from p_pawmil.f_train import evaluate_model
from p_pawmil.f_train import evaluate_model_weighted
from p_pawmil.f_train import get_auc_multilabel
from p_pawmil.f_train import get_auc_multilabel_grouped
from p_pawmil.f_train import grouped_stratified_kfold_split
from p_pawmil.f_utils import subset_list as sl
from p_pawmil.f_utils import format_label_file 

##
import torch
from torch.utils.data import DataLoader


In [215]:
###############
## FUNCTIONS ##
###############

def bootstrap_auc(true, pred, n_boot=2000):
    true = np.asarray(true)
    pred = np.asarray(pred)

    n = true.shape[0]
    aucs = []

    for _ in range(n_boot):
        idx = np.random.choice(n, n, replace=True)
        aucs.append(get_auc_multilabel(true[idx], pred[idx]))

    aucs = np.asarray(aucs)

    return {
        "auc": get_auc_multilabel(true, pred),
        "ci_low": np.percentile(aucs, 2.5, axis=0),
        "ci_high": np.percentile(aucs, 97.5, axis=0),
    }

def bootstrap_auc_diff(true, pred_a, pred_b, n_boot=2000):
    true = np.asarray(true)
    pred_a = np.asarray(pred_a)
    pred_b = np.asarray(pred_b)

    n = true.shape[0]
    diffs = []

    for _ in range(n_boot):
        idx = np.random.choice(n, n, replace=True)

        auc_a = np.asarray(get_auc_multilabel(true[idx], pred_a[idx]))
        auc_b = np.asarray(get_auc_multilabel(true[idx], pred_b[idx]))

        diffs.append(auc_b - auc_a)

    diffs = np.asarray(diffs)

    return {
        "diff": np.asarray(get_auc_multilabel(true, pred_b))
               - np.asarray(get_auc_multilabel(true, pred_a)),
        "ci_low": np.percentile(diffs, 2.5, axis=0),
        "ci_high": np.percentile(diffs, 97.5, axis=0),
        "bootstrap_diffs": diffs
    }

def stratified_bootstrap_auc_diff(true, pred_a, pred_b, n_boot=2000):
    true = np.asarray(true)
    pred_a = np.asarray(pred_a)
    pred_b = np.asarray(pred_b)

    n = len(true)

    pos_idx = np.where(true[:, 0] == 1)[0]
    neg_idx = np.where(true[:, 0] == 0)[0]

    diffs = []

    for _ in range(n_boot):
        
        boot_pos = np.random.choice(pos_idx, size=len(pos_idx), replace=True)
        boot_neg = np.random.choice(neg_idx, size=len(neg_idx), replace=True)
        idx = np.concatenate([boot_pos, boot_neg])

        auc_a = np.asarray(get_auc_multilabel(true[idx], pred_a[idx]))
        auc_b = np.asarray(get_auc_multilabel(true[idx], pred_b[idx]))

        diffs.append(auc_b - auc_a)

    diffs = np.asarray(diffs)

    return {
        "diff": np.asarray(get_auc_multilabel(true, pred_b))
               - np.asarray(get_auc_multilabel(true, pred_a)),
        "ci_low": np.percentile(diffs, 2.5, axis=0),
        "ci_high": np.percentile(diffs, 97.5, axis=0),
        "bootstrap_diffs": diffs
    }

#
###

In [263]:
###################
## GENERAL SETUP ##
###################

## User inputs
DEVICE = "mps"
COHORT = 'icgc-c' 
# COHORT = 'tcga-oct'
RUN_IDENTIFIER = "_1000" # enter "" by default
PREDICTION_LEVEL = "group-level" # "sample-level" or "group-level"
APPLY_GROUP_WEIGHTS = False
AGGREGATOR = 'clam' # for AGGREGATOR in ['pawmil', 'clam']:
SELECTED_LABEL_COLUMN = 'relapse'
# SELECTED_LABEL_COLUMN = 'EMT.GAIN'
# SELECTED_LABEL_COLUMN = 'MYC.GAIN'
# ALL_LABELS_COLUMNS = [
    # 'relapse', # For icgc-c
    # 'EMT.GAIN', # For tcga-oct
    # 'MYC.GAIN', # For tcga-oct
# ]

## CSV variables (ICGCC)
if COHORT == 'icgc-c':
    pti = '/Volumes/Elements/BDI/projects/GSG/metadata/icgcc-relapse/metadata_ICGCCRelapse.csv' # ICGCC 
    path_column = 'path_to_image_1' # ICGCC
    case_id_column = 'slide_id' # ICGCC # 'sanger_case_id' for patient-level or 'slide_id' for sample-level predictions
    weight_group_column = 'sanger_case_id' # 'sanger_case_id' # ICGCC

## CSV variables (TCGA)
if COHORT == 'tcga-oct':
    pti = '/Volumes/Elements/BDI/projects/GSG/metadata/tcga-cnv/master-sheet-2025-04-18.csv' # TCGA
    path_column = 'wsi_file_path_local' # TCGA
    case_id_column = 'sample_id_local' # TCGA
    weight_group_column = '' # TCGA

MODEL_NAME_LIST = [    
    'model_resnet_icgcc_baseline',
    'model_resnet_icgcc_raw_FTL_0',
    'model_resnet50_icgcc_baseline',
    'model_resnet50_icgcc_raw_FTL_0', 
    'model_vit_icgcc_baseline',
    'model_vit_icgcc_raw_FTL_0', 
    #
    # 'model_resnet_tcgaoct_baseline',
    # 'model_resnet_tcgaoct_raw_FTL_0',
    # 'model_resnet50_tcgaoct_baseline',
    # 'model_resnet50_tcgaoct_raw_FTL_0',
    # 'model_vit_tcgaoct_baseline',
    # 'model_vit_tcgaoct_raw_FTL_0',
    #
    # 'model_resnet_icgcc_raw_FTL_0',
    # 'model_resnet_icgcc_raw_FTL_25',
    #
    # 'model_resnet_tcgaoct_raw_FTL_0',
    # 'model_resnet_tcgaoct_raw_FTL_1',
    # 'model_resnet_tcgaoct_raw_FTL_1',
    # 'model_resnet_tcgaoct_raw_FTL_3',
    # 'model_resnet_tcgaoct_raw_FTL_3',
    # 'model_resnet_tcgaoct_raw_FTL_5',
    # 'model_resnet_tcgaoct_raw_FTL_5',
    # 'model_resnet_tcgaoct_raw_FTL_25',
    ]

## Load the CSV file
df = pd.read_csv(pti)
print(df.shape)

## Import aggregator
if AGGREGATOR == 'clam': 
    from p_pawmil.f_aggregator_clam import CLAM_SB as AggregatorArchitecture
    
if AGGREGATOR == 'pawmil': 
    from p_pawmil.f_aggregator_pawmil import AttentionLastAggregator as AggregatorArchitecture

#
###

(393, 44)


In [267]:
#######################
## MODEL PREDICTIONS ##
#######################

all_singlefold_results_list = []
all_multifold_results_list = []
for model_pair_index in range(0,len(MODEL_NAME_LIST)-1,2):

    ## Select model couple
    MODEL_NAME_A = MODEL_NAME_LIST[model_pair_index] # 'model_resnet_icgcc_baseline'
    MODEL_NAME_B = MODEL_NAME_LIST[model_pair_index+1] # 'model_resnet_icgcc_raw_FTL_0'

    ## Check 
    print(f"{MODEL_NAME_A} vs {MODEL_NAME_B}")

    multifold_results_list = []
    multifold_groundtruth_list = []
    multifold_predictions_list = []
    for SELECTED_TESTING_FOLD in [1, 2, 3, 4, 5]:
    
        singlefold_groundtruth_list = []
        singlefold_predictions_list = []
        for MODEL_NAME in [MODEL_NAME_A, MODEL_NAME_B]:
        
            #################
            ## MODEL SETUP ##
            #################
            
            # MODEL_NAME = 'model_vit_tcgaoct_baseline' # for MODEL_NAME in MODEL_NAME_LIST:
            # MODEL_NAME = 'model_vit_tcgaoct_raw_FTL_0' # for MODEL_NAME in MODEL_NAME_LIST:    
            
            ## Define the paths to the input files
            pt_embeddings_file = f"/Volumes/Elements/BDI/projects/GSG/outputs/o3_mil/{COHORT}/{MODEL_NAME}/encoder/obj_embeddings.pkl"
            PTI = f"/Volumes/Elements/BDI/projects/GSG/outputs/o3_mil/{COHORT}/{MODEL_NAME}/"
            PTO = f"/Volumes/Elements/BDI/projects/GSG/outputs/o3_mil_bootstrap/{COHORT}/{MODEL_NAME}/"
            if os.path.exists(PTO) == False:
                os.makedirs(PTO, exist_ok=True)
                print(PTO)
            
            # for SELECTED_LABEL_COLUMN in ALL_LABELS_COLUMNS:
            
            ## Create output directory
            pti = PTI + f"aggregator_{AGGREGATOR}{RUN_IDENTIFIER}/singletask-{SELECTED_LABEL_COLUMN}/"
            pto = PTO + f"aggregator_{AGGREGATOR}{RUN_IDENTIFIER}/singletask-{SELECTED_LABEL_COLUMN}/"
            if os.path.exists(pto) == False:
                os.makedirs(pto, exist_ok=True)
                print(pto)
            
            ## Extract the relevant columns
            labels = df[[SELECTED_LABEL_COLUMN]]
            paths = df[[path_column]]
            case_ids = df[[case_id_column]]
            
            ## Threshold the labels to binarize them
            labels = (labels > 0.0).astype(int)
            
            ## Repeat each row n times to match the augmented dataset
            n = 1
            labels_repeated = pd.concat([labels] * n, ignore_index=True)
            paths_repeated = pd.concat([paths] * n, ignore_index=True)
            case_ids_repeated = pd.concat([case_ids] * n, ignore_index=True)       
            
            ## Save the extracted and repeated columns to separate text files with tab separation
            pt_labels_file = f"{pto}/tab_labels.txt"
            pt_paths_file = f"{pto}/tab_paths.txt"
            pt_groups_file = f"{pto}/tab_groups.txt"
            labels_repeated.to_csv(pt_labels_file, index=False, header=False, sep='\t')
            paths_repeated.to_csv(pt_paths_file, index=False, header=False, sep='\t')
            case_ids_repeated.to_csv(pt_groups_file, index=False, header=False, sep='\t')
            
            ## Group weighting during training
            if weight_group_column != '':
                weight_groups = df[[weight_group_column]]
                weight_groups_repeated = pd.concat([weight_groups] * n, ignore_index=True)
                pt_weight_groups_file = f"{pto}/tab_groups_2.txt"
                weight_groups_repeated.to_csv(pt_weight_groups_file, index=False, header=False, sep='\t')
            else:
                pt_weight_groups_file = ""
              
            ## Hypertrain aggregator
            pt_output_folder = pti
            device = DEVICE
            
            ## Paths
            pt_folder_out = pt_output_folder
            pt_subfolder_out = pt_folder_out + f'hypertrain_fold_{SELECTED_TESTING_FOLD}/'
            pt_models_folder = pt_subfolder_out + "models/"
            
            ## Parameters
            selected_testing_fold = SELECTED_TESTING_FOLD - 1 # /!\ Do not change this line
            
            ## Get labels and groups and embeddings
            labels = format_label_file(pt_labels_file)
            groups = np.genfromtxt(pt_groups_file, dtype="str")
            embeddings = pickle.load(open(pt_embeddings_file, "rb"))
            if pt_weight_groups_file != "":
                weight_groups = np.genfromtxt(pt_weight_groups_file, dtype="str")
            
            ## Fixed parameters
            num_folds = 5
            num_hidden = 256
            num_repeats = 3
            
            ## Dependent parameters
            num_classes = len(labels[0])
            num_embeddings = embeddings[0].shape[1]
        
            ##########################
            ## PREPARE TESTING DATA ##
            ##########################
            
            ## Determine test fold indices 
            trainval_folds, test_folds = grouped_stratified_kfold_split(labels, groups, num_folds)
            test_index = test_folds[selected_testing_fold-1] # /!\ Extra shift in test_index label—no consequence if kept in place
            
            ## Get test labels and embeddings
            test_labels = sl(labels, test_index)
            test_embeddings = sl(embeddings, test_index)
            test_groups  = sl(weight_groups, test_index)
            
            ## Create dataset and dataloader
            test_dataset = InstanceEmbeddingsDataset(test_embeddings, test_labels, num_classes=num_classes)
            test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)        
            
            #
            ###
        
            #########################
            ## EVALUATE AGGREGATOR ##
            #########################
            
            ## Get best parameters
            all_train_data, all_val_data, all_test_data, _, best_hyperparams = collect_results_across_labels(master_folder = pt_output_folder, selected_folders = [''], column_indices = [2, 3, 4])
            batch_size, num_epochs, learn_rate = best_hyperparams[0][selected_testing_fold].split("_")
            batch_size = int(float(batch_size))
            num_epochs = int(float(num_epochs))
            learn_rate = float(learn_rate)
            
            ## Collectors
            outputs_list = []
            true_list = []
            
            ## For each fold
            for selected_training_fold in range(num_folds):
            
                ## For each repeat
                for repeat in range(num_repeats):
            
                    ## Model id
                    model_id = f"mod_agg_fold{selected_training_fold}_repeat{repeat}_bs{batch_size}_ne{num_epochs}_lr{learn_rate}.pth"
            
                    ## Check if checkpoint exists
                    if os.path.exists(pt_models_folder + model_id) == True:
            
                        ## Evaluate at sample-level            
                        pt_aggregator_in = pt_models_folder + model_id
                        
                        ## Instantiate the model
                        aggregator = AggregatorArchitecture(num_embeddings, num_classes, num_hidden)
                        if os.path.exists(pt_aggregator_in):
                            aggregator.load_state_dict(torch.load(pt_aggregator_in))
                        else: 
                            print("model not found")
                            break
                        model = aggregator
                    
                        ## Evaluate
                        outputs, _, _, true = evaluate_model(model, test_loader, device)
            
                        ## Collect
                        outputs_list += [outputs]
                        true_list += [true]
            
            #
            ###
            
            ##############################
            ## MEAN ENSEMBLE PREDICTION ##
            ##############################
        
            ## Average predictions
            outputs_mean = np.stack(outputs_list).mean(0)
            # print(get_auc_multilabel(true,outputs_mean))
    
            ## Aggregate predictions by group
            if PREDICTION_LEVEL == "group-level":
    
                ## Rename variables
                groups = test_groups.copy()
                y_true = true.copy()
                y_scores = outputs_mean.copy()
                
                ## Get unique groups
                groups = np.array(groups)
                unique_groups = np.unique(groups)
                
                ## Aggregate labels to group level
                y_true_grouped = np.array([
                    y_true[groups == group].mean(axis=0)
                    for group in unique_groups
                ])
            
                ## Aggregate predictions
                y_scores_grouped = np.array([
                    y_scores[groups == group].mean(axis=0)
                    for group in unique_groups
                ])
            
                ## Rename
                true = y_true_grouped.copy()
                outputs_mean = y_scores_grouped.copy()            # print(get_auc_multilabel(true,outputs_mean))
        
            ## Collect
            singlefold_predictions_list += [outputs_mean]
            singlefold_groundtruth_list += [true]

            """
            result = bootstrap_auc(
                true,
                outputs_mean,    
                n_boot=2000
            )
            
            print("AUC:", result["auc"])
            print("95% CI:", result["ci_low"], result["ci_high"])            
            """
    
            #
            ###
    
        #########################
        ## COLLECT PREDICTIONS ##
        #########################
        
        multifold_predictions_list += [singlefold_predictions_list]
        multifold_groundtruth_list += [singlefold_groundtruth_list]
    
        #
        ###
        
    ###############################
    ## BOOTSTRAP AUC DIFFERENCES ##
    ###############################
    
    ## Single test fold ∆AUCs
    singlefold_results_list = []
    bootstrap_diffs_all = []
    for i in range(0,5):
        
        true_A = multifold_groundtruth_list[i][0]
        true_B = multifold_groundtruth_list[i][1]
        if(((true_A != true_B)*1).sum() > 0): # Sanity check: no differences in labels
            print("Error: labels A and B differ")
            break
        
        outputs_mean_A = multifold_predictions_list[i][0]
        outputs_mean_B = multifold_predictions_list[i][1]

        if PREDICTION_LEVEL == "group-level":
            result = stratified_bootstrap_auc_diff(
                true_A,
                outputs_mean_A,
                outputs_mean_B,
                n_boot=10000
            )
        else:
            result = bootstrap_auc_diff(
                true_A,
                outputs_mean_A,
                outputs_mean_B,
                n_boot=10000
            )
        
        print("Delta AUC:", result["diff"])
        print("95% CI:", result["ci_low"], result["ci_high"])

        ## Collect
        singlefold_results_list += [{
            "fold": i, 
            "diff_AUC": result["diff"], 
            "95CI_lo": result["ci_low"], 
            "95CI_hi": result["ci_high"]
        }]
    
        ## Need the bootstrap distribution itself
        bootstrap_diffs_all.append(result["bootstrap_diffs"])
    
    ## Multifold ∆AUCs
    bootstrap_diffs_all = np.asarray(bootstrap_diffs_all)
    
    ## Shape: (5 folds, n_boot, n_labels)
    multifold_bootstrap_diffs = bootstrap_diffs_all.mean(axis=0)
    
    ## Observed mean Delta AUC across folds
    mean_diff = np.mean(
        [x["diff_AUC"] for x in singlefold_results_list],
        axis=0
    )
    
    ## 95% bootstrap CI
    ci_low = np.percentile(multifold_bootstrap_diffs, 2.5, axis=0)
    ci_high = np.percentile(multifold_bootstrap_diffs, 97.5, axis=0)

    ## Collect
    multifold_results_list += [{
        "diff_AUC": mean_diff,
        "95CI_lo": ci_low,
        "95CI_hi": ci_high
    }]

    #
    ###

    #################
    ## COLLECT ALL ##
    #################

    print(multifold_results_list)
    for i in range(0,5): print(singlefold_results_list[i])

    all_multifold_results_list += [multifold_results_list]
    all_singlefold_results_list += [all_singlefold_results_list]

    #
    ###

model_resnet_icgcc_baseline vs model_resnet_icgcc_raw_FTL_0
Processing label: 
Processing fold 1...
Best hyperparameter set for fold 1 is: 4
HP star is: 1.0_8.0_0.001
Processing fold 2...
Best hyperparameter set for fold 2 is: 4
HP star is: 1.0_8.0_0.001
Processing fold 3...
Best hyperparameter set for fold 3 is: 4
HP star is: 1.0_8.0_0.001
Processing fold 4...
Best hyperparameter set for fold 4 is: 4
HP star is: 1.0_8.0_0.001
Processing fold 5...
Best hyperparameter set for fold 5 is: 4
HP star is: 1.0_8.0_0.001


/var/folders/tb/4hh90ttd1nl4xf1zbfyq_n8w0000gp/T/ipykernel_97837/3194987045.py:161: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  aggregator.load_state_dict(torch.load(pt_ag

Processing label: 
Processing fold 1...
Best hyperparameter set for fold 1 is: 3
HP star is: 1.0_8.0_0.0001
Processing fold 2...
Best hyperparameter set for fold 2 is: 3
HP star is: 1.0_8.0_0.0001
Processing fold 3...
Best hyperparameter set for fold 3 is: 3
HP star is: 1.0_8.0_0.0001
Processing fold 4...
Best hyperparameter set for fold 4 is: 3
HP star is: 1.0_8.0_0.0001
Processing fold 5...
Best hyperparameter set for fold 5 is: 3
HP star is: 1.0_8.0_0.0001


/var/folders/tb/4hh90ttd1nl4xf1zbfyq_n8w0000gp/T/ipykernel_97837/3194987045.py:161: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  aggregator.load_state_dict(torch.load(pt_ag

Processing label: 
Processing fold 1...
Best hyperparameter set for fold 1 is: 4
HP star is: 1.0_8.0_0.001
Processing fold 2...
Best hyperparameter set for fold 2 is: 4
HP star is: 1.0_8.0_0.001
Processing fold 3...
Best hyperparameter set for fold 3 is: 4
HP star is: 1.0_8.0_0.001
Processing fold 4...
Best hyperparameter set for fold 4 is: 4
HP star is: 1.0_8.0_0.001
Processing fold 5...
Best hyperparameter set for fold 5 is: 4
HP star is: 1.0_8.0_0.001


...

Processing label: 
Processing fold 1...
Best hyperparameter set for fold 1 is: 3
HP star is: 1.0_8.0_0.0001
Processing fold 2...
Best hyperparameter set for fold 2 is: 3
HP star is: 1.0_8.0_0.0001
Processing fold 3...
Best hyperparameter set for fold 3 is: 3
HP star is: 1.0_8.0_0.0001
Processing fold 4...
Best hyperparameter set for fold 4 is: 3
HP star is: 1.0_8.0_0.0001
Processing fold 5...
Best hyperparameter set for fold 5 is: 3
HP star is: 1.0_8.0_0.0001


/var/folders/tb/4hh90ttd1nl4xf1zbfyq_n8w0000gp/T/ipykernel_97837/3194987045.py:161: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  aggregator.load_state_dict(torch.load(pt_ag

Processing label: 
Processing fold 1...
Best hyperparameter set for fold 1 is: 4
HP star is: 1.0_8.0_0.001
Processing fold 2...
Best hyperparameter set for fold 2 is: 4
HP star is: 1.0_8.0_0.001
Processing fold 3...
Best hyperparameter set for fold 3 is: 4
HP star is: 1.0_8.0_0.001
Processing fold 4...
Best hyperparameter set for fold 4 is: 3
HP star is: 1.0_8.0_0.0001
Processing fold 5...
Best hyperparameter set for fold 5 is: 4
HP star is: 1.0_8.0_0.001


/var/folders/tb/4hh90ttd1nl4xf1zbfyq_n8w0000gp/T/ipykernel_97837/3194987045.py:161: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  aggregator.load_state_dict(torch.load(pt_ag

Delta AUC: [-0.08]
95% CI: [-0.17555556] [0.00444444]
Delta AUC: [0.02083333]
95% CI: [-0.07150298] [0.12797619]
Delta AUC: [-0.1]
95% CI: [-0.19428571] [-0.02571429]
Delta AUC: [-0.08163265]
95% CI: [-0.21282799] [0.02623907]
Delta AUC: [-0.03714286]
95% CI: [-0.14] [0.05428571]
[{'diff_AUC': array([-0.05558844]), '95CI_lo': array([-0.10258681]), '95CI_hi': array([-0.01264922])}]
{'fold': 0, 'diff_AUC': array([-0.08]), '95CI_lo': array([-0.17555556]), '95CI_hi': array([0.00444444])}
{'fold': 1, 'diff_AUC': array([0.02083333]), '95CI_lo': array([-0.07150298]), '95CI_hi': array([0.12797619])}
{'fold': 2, 'diff_AUC': array([-0.1]), '95CI_lo': array([-0.19428571]), '95CI_hi': array([-0.02571429])}
{'fold': 3, 'diff_AUC': array([-0.08163265]), '95CI_lo': array([-0.21282799]), '95CI_hi': array([0.02623907])}
{'fold': 4, 'diff_AUC': array([-0.03714286]), '95CI_lo': array([-0.14]), '95CI_hi': array([0.05428571])}


In [262]:
print(PREDICTION_LEVEL)
print(SELECTED_LABEL_COLUMN)
for model_pair_index in range(0,len(MODEL_NAME_LIST)-1,2):    
    MODEL_NAME_A = MODEL_NAME_LIST[model_pair_index]
    MODEL_NAME_B = MODEL_NAME_LIST[model_pair_index+1]    
    print(f"{MODEL_NAME_B} compared to {MODEL_NAME_A}")
for _ in all_multifold_results_list: print(_)

sample-level
MYC.GAIN
model_resnet_tcgaoct_raw_FTL_1 compared to model_resnet_tcgaoct_raw_FTL_0
model_resnet_tcgaoct_raw_FTL_3 compared to model_resnet_tcgaoct_raw_FTL_1
model_resnet_tcgaoct_raw_FTL_5 compared to model_resnet_tcgaoct_raw_FTL_3
model_resnet_tcgaoct_raw_FTL_25 compared to model_resnet_tcgaoct_raw_FTL_5
[{'diff_AUC': array([0.02583622]), '95CI_lo': array([-0.01832956]), '95CI_hi': array([0.06957472])}]
[{'diff_AUC': array([-0.06614239]), '95CI_lo': array([-0.12814513]), '95CI_hi': array([-0.00515388])}]
[{'diff_AUC': array([-0.05698857]), '95CI_lo': array([-0.10631742]), '95CI_hi': array([-0.00488025])}]
[{'diff_AUC': array([-0.00698333]), '95CI_lo': array([-0.07677941]), '95CI_hi': array([0.06263615])}]


In [259]:
print(PREDICTION_LEVEL)
print(SELECTED_LABEL_COLUMN)
for model_pair_index in range(0,len(MODEL_NAME_LIST)-1,2):    
    MODEL_NAME_A = MODEL_NAME_LIST[model_pair_index]
    MODEL_NAME_B = MODEL_NAME_LIST[model_pair_index+1]    
    print(f"{MODEL_NAME_B} compared to {MODEL_NAME_A}")
for _ in all_multifold_results_list: print(_)

sample-level
EMT.GAIN
model_resnet_tcgaoct_raw_FTL_1 compared to model_resnet_tcgaoct_raw_FTL_0
model_resnet_tcgaoct_raw_FTL_3 compared to model_resnet_tcgaoct_raw_FTL_1
model_resnet_tcgaoct_raw_FTL_5 compared to model_resnet_tcgaoct_raw_FTL_3
model_resnet_tcgaoct_raw_FTL_25 compared to model_resnet_tcgaoct_raw_FTL_5
[{'diff_AUC': array([0.01269048]), '95CI_lo': array([-0.03208042]), '95CI_hi': array([0.05908028])}]
[{'diff_AUC': array([-0.02645238]), '95CI_lo': array([-0.0781405]), '95CI_hi': array([0.02294153])}]
[{'diff_AUC': array([-0.0455]), '95CI_lo': array([-0.08198041]), '95CI_hi': array([-0.00952086])}]
[{'diff_AUC': array([-0.03097619]), '95CI_lo': array([-0.09445806]), '95CI_hi': array([0.03211155])}]


In [256]:
print(PREDICTION_LEVEL)
print(SELECTED_LABEL_COLUMN)
for model_pair_index in range(0,len(MODEL_NAME_LIST)-1,2):    
    MODEL_NAME_A = MODEL_NAME_LIST[model_pair_index]
    MODEL_NAME_B = MODEL_NAME_LIST[model_pair_index+1]    
    print(f"{MODEL_NAME_B} compared to {MODEL_NAME_A}")
for _ in all_multifold_results_list: print(_)

sample-level
relapse
model_resnet_icgcc_raw_FTL_25 compared to model_resnet_icgcc_raw_FTL_0
[{'diff_AUC': array([0.01100475]), '95CI_lo': array([-0.02957377]), '95CI_hi': array([0.0526851])}]


In [253]:
print(PREDICTION_LEVEL)
print(SELECTED_LABEL_COLUMN)
for model_pair_index in range(0,len(MODEL_NAME_LIST)-1,2):    
    MODEL_NAME_A = MODEL_NAME_LIST[model_pair_index]
    MODEL_NAME_B = MODEL_NAME_LIST[model_pair_index+1]    
    print(f"{MODEL_NAME_B} compared to {MODEL_NAME_A}")
for _ in all_multifold_results_list: print(_)

sample-level
MYC.GAIN
model_resnet_tcgaoct_raw_FTL_0 compared to model_resnet_tcgaoct_baseline
model_resnet50_tcgaoct_raw_FTL_0 compared to model_resnet50_tcgaoct_baseline
model_vit_tcgaoct_raw_FTL_0 compared to model_vit_tcgaoct_baseline
[{'diff_AUC': array([0.10019922]), '95CI_lo': array([0.00507162]), '95CI_hi': array([0.19302824])}]
[{'diff_AUC': array([0.10969907]), '95CI_lo': array([0.02868595]), '95CI_hi': array([0.19186509])}]
[{'diff_AUC': array([0.01872706]), '95CI_lo': array([-0.05099945]), '95CI_hi': array([0.08897452])}]


In [250]:
print(PREDICTION_LEVEL)
print(SELECTED_LABEL_COLUMN)
for model_pair_index in range(0,len(MODEL_NAME_LIST)-1,2):    
    MODEL_NAME_A = MODEL_NAME_LIST[model_pair_index]
    MODEL_NAME_B = MODEL_NAME_LIST[model_pair_index+1]    
    print(f"{MODEL_NAME_B} compared to {MODEL_NAME_A}")
for _ in all_multifold_results_list: print(_)

sample-level
EMT.GAIN
model_resnet_tcgaoct_raw_FTL_0 compared to model_resnet_tcgaoct_baseline
model_resnet50_tcgaoct_raw_FTL_0 compared to model_resnet50_tcgaoct_baseline
model_vit_tcgaoct_raw_FTL_0 compared to model_vit_tcgaoct_baseline
[{'diff_AUC': array([0.05783333]), '95CI_lo': array([-0.03749672]), '95CI_hi': array([0.15424587])}]
[{'diff_AUC': array([0.1037619]), '95CI_lo': array([0.01306585]), '95CI_hi': array([0.19613656])}]
[{'diff_AUC': array([0.00254762]), '95CI_lo': array([-0.0781362]), '95CI_hi': array([0.08510397])}]


In [268]:
print(PREDICTION_LEVEL)
print(SELECTED_LABEL_COLUMN)
for model_pair_index in range(0,len(MODEL_NAME_LIST)-1,2):    
    MODEL_NAME_A = MODEL_NAME_LIST[model_pair_index]
    MODEL_NAME_B = MODEL_NAME_LIST[model_pair_index+1]    
    print(f"{MODEL_NAME_B} compared to {MODEL_NAME_A}")
for _ in all_multifold_results_list: print(_)

group-level
relapse
model_resnet_icgcc_raw_FTL_0 compared to model_resnet_icgcc_baseline
model_resnet50_icgcc_raw_FTL_0 compared to model_resnet50_icgcc_baseline
model_vit_icgcc_raw_FTL_0 compared to model_vit_icgcc_baseline
[{'diff_AUC': array([-0.06455572]), '95CI_lo': array([-0.13113883]), '95CI_hi': array([-0.00349487])}]
[{'diff_AUC': array([-0.06883706]), '95CI_lo': array([-0.15169748]), '95CI_hi': array([0.00918733])}]
[{'diff_AUC': array([-0.05558844]), '95CI_lo': array([-0.10258681]), '95CI_hi': array([-0.01264922])}]


In [244]:
print(PREDICTION_LEVEL)
print(SELECTED_LABEL_COLUMN)
for model_pair_index in range(0,len(MODEL_NAME_LIST)-1,2):    
    MODEL_NAME_A = MODEL_NAME_LIST[model_pair_index]
    MODEL_NAME_B = MODEL_NAME_LIST[model_pair_index+1]    
    print(f"{MODEL_NAME_B} compared to {MODEL_NAME_A}")
for _ in all_multifold_results_list: print(_)

sample-level
relapse
model_resnet_icgcc_raw_FTL_0 compared to model_resnet_icgcc_baseline
model_resnet50_icgcc_raw_FTL_0 compared to model_resnet50_icgcc_baseline
model_vit_icgcc_raw_FTL_0 compared to model_vit_icgcc_baseline
[{'diff_AUC': array([-0.01078431]), '95CI_lo': array([-0.04791748]), '95CI_hi': array([0.02529729])}]
[{'diff_AUC': array([-0.02535767]), '95CI_lo': array([-0.06849238]), '95CI_hi': array([0.01514528])}]
[{'diff_AUC': array([-0.019599]), '95CI_lo': array([-0.04511743]), '95CI_hi': array([0.00398678])}]


In [ ]:
relapse
model_resnet_icgcc_raw_FTL_0 compared to model_resnet_icgcc_baseline
[{'diff_AUC': array([-0.01078431]), '95CI_lo': array([-0.04791748]), '95CI_hi': array([0.02529729])}]
model_resnet50_icgcc_raw_FTL_0 compared to model_resnet50_icgcc_baseline
[{'diff_AUC': array([-0.02535767]), '95CI_lo': array([-0.06849238]), '95CI_hi': array([0.01514528])}]
model_vit_icgcc_raw_FTL_0 compared to model_vit_icgcc_baseline
[{'diff_AUC': array([-0.019599]), '95CI_lo': array([-0.04511743]), '95CI_hi': array([0.00398678])}]

EMT.GAIN
model_resnet_tcgaoct_raw_FTL_0 compared to model_resnet_tcgaoct_baseline
[{'diff_AUC': array([0.05783333]), '95CI_lo': array([-0.03749672]), '95CI_hi': array([0.15424587])}]
model_resnet50_tcgaoct_raw_FTL_0 compared to model_resnet50_tcgaoct_baseline
[{'diff_AUC': array([0.1037619]), '95CI_lo': array([0.01306585]), '95CI_hi': array([0.19613656])}]
model_vit_tcgaoct_raw_FTL_0 compared to model_vit_tcgaoct_baseline
[{'diff_AUC': array([0.00254762]), '95CI_lo': array([-0.0781362]), '95CI_hi': array([0.08510397])}]

MYC.GAIN
model_resnet_tcgaoct_raw_FTL_0 compared to model_resnet_tcgaoct_baseline
[{'diff_AUC': array([0.10019922]), '95CI_lo': array([0.00507162]), '95CI_hi': array([0.19302824])}]
model_resnet50_tcgaoct_raw_FTL_0 compared to model_resnet50_tcgaoct_baseline
[{'diff_AUC': array([0.10969907]), '95CI_lo': array([0.02868595]), '95CI_hi': array([0.19186509])}]
model_vit_tcgaoct_raw_FTL_0 compared to model_vit_tcgaoct_baseline
[{'diff_AUC': array([0.01872706]), '95CI_lo': array([-0.05099945]), '95CI_hi': array([0.08897452])}]

In [ ]:
relapse
model_resnet_icgcc_raw_FTL_25 compared to model_resnet_icgcc_raw_FTL_0
[{'diff_AUC': array([0.01100475]), '95CI_lo': array([-0.02957377]), '95CI_hi': array([0.0526851])}]

EMT.GAIN
model_resnet_tcgaoct_raw_FTL_1 compared to model_resnet_tcgaoct_raw_FTL_0
[{'diff_AUC': array([0.01269048]), '95CI_lo': array([-0.03208042]), '95CI_hi': array([0.05908028])}]
model_resnet_tcgaoct_raw_FTL_3 compared to model_resnet_tcgaoct_raw_FTL_1
[{'diff_AUC': array([-0.02645238]), '95CI_lo': array([-0.0781405]), '95CI_hi': array([0.02294153])}]
model_resnet_tcgaoct_raw_FTL_5 compared to model_resnet_tcgaoct_raw_FTL_3
[{'diff_AUC': array([-0.0455]), '95CI_lo': array([-0.08198041]), '95CI_hi': array([-0.00952086])}]
model_resnet_tcgaoct_raw_FTL_25 compared to model_resnet_tcgaoct_raw_FTL_5
[{'diff_AUC': array([-0.03097619]), '95CI_lo': array([-0.09445806]), '95CI_hi': array([0.03211155])}]

MYC.GAIN
model_resnet_tcgaoct_raw_FTL_1 compared to model_resnet_tcgaoct_raw_FTL_0
[{'diff_AUC': array([0.02583622]), '95CI_lo': array([-0.01832956]), '95CI_hi': array([0.06957472])}]
model_resnet_tcgaoct_raw_FTL_3 compared to model_resnet_tcgaoct_raw_FTL_1
[{'diff_AUC': array([-0.06614239]), '95CI_lo': array([-0.12814513]), '95CI_hi': array([-0.00515388])}]
model_resnet_tcgaoct_raw_FTL_5 compared to model_resnet_tcgaoct_raw_FTL_3
[{'diff_AUC': array([-0.05698857]), '95CI_lo': array([-0.10631742]), '95CI_hi': array([-0.00488025])}]
model_resnet_tcgaoct_raw_FTL_25 compared to model_resnet_tcgaoct_raw_FTL_5
[{'diff_AUC': array([-0.00698333]), '95CI_lo': array([-0.07677941]), '95CI_hi': array([0.06263615])}]

In [ ]:
relapse
model_resnet_icgcc_raw_FTL_0 compared to model_resnet_icgcc_baseline
model_resnet50_icgcc_raw_FTL_0 compared to model_resnet50_icgcc_baseline
model_vit_icgcc_raw_FTL_0 compared to model_vit_icgcc_baseline
[{'diff_AUC': array([-0.06455572]), '95CI_lo': array([-0.13113883]), '95CI_hi': array([-0.00349487])}]
[{'diff_AUC': array([-0.06883706]), '95CI_lo': array([-0.15169748]), '95CI_hi': array([0.00918733])}]
[{'diff_AUC': array([-0.05558844]), '95CI_lo': array([-0.10258681]), '95CI_hi': array([-0.01264922])}]